<a href="https://colab.research.google.com/github/HoudiniJezewskii/ai-medical-support-chatbot/blob/main/notebooks/00_signora_Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

In [ ]:
from huggingface_hub import logout, login

logout()
login()

In [ ]:
 # load medgemma 4 bit
import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)

model_name = "google/medgemma-4b-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(model_name)

model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("MedGemma loaded successfully.")

In [ ]:
def ask_medgemma(question, conversation_history):
    system_instruction = """
You are a concise medical information chatbot.

Rules:
- Answer in no more than 3 short sentences.
- Do not use bullet points or headings.
- Do not say "Okay" or repeat the question.
- Do not diagnose.
- Do not prescribe any medication.
- Do not provide extra home remedies.
- If serious symptoms are mentioned, recommend immediate medical care.
- State that the information is general education.
"""

# initial message, set chatbots rule n instruction

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_instruction
                }
            ]
        }
    ]

    # Keep only the 4 most recent conversation messages (maybe change later?)
    # this is to give the bot short-term memory
    messages.extend(conversation_history[-4:])

    # add users new question to message list
    messages.append(
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": question
                }
            ]
        }
    )

    # convert messages into the format MedGemma expects

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    # generate answer
    # 70 tokens for now to keep answer short + faster
    outputs = model.generate(
        **inputs,
        max_new_tokens=70,
        do_sample=False,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
        pad_token_id=processor.tokenizer.eos_token_id
    )

    answer_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    # convert generated tokens into text
    answer = processor.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


def start_chatbot():
    print("AI Medical Support Chatbot")
    print("Type 'quit' to exit.\n")

    conversation_history = []

    while True:
        question = input("You: ").strip()

        if question.lower() in ["quit", "exit", "bye"]:
            print("Chatbot: Goodbye!")
            break

        if not question:
            continue

        answer = ask_medgemma(
            question,
            conversation_history
        )

        print("\nChatbot:")
        print(answer)
        print()

        # save users question in convo history AFTER generating answer
        conversation_history.append(
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": question
                    }
                ]
            }
        )

        # save bots answer
        conversation_history.append(
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": answer
                    }
                ]
            }
        )



In [ ]:
start_chatbot()